In [1]:
import pandas as pd
from tensorflow import config
import matplotlib.pyplot as plt
import tensorflow as tf
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
import gc
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.mixed_precision import set_global_policy
set_global_policy("mixed_float16")

gpus = config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        config.experimental.set_visible_devices(gpus[0], 'GPU')
        config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)

INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce GTX 1650, compute capability 7.5


In [2]:
def show_loss(hist):
    loss     = hist.history['loss']
    val_loss = hist.history['val_loss']
    epochs   = range(len(loss))
    plt.figure()
    plt.plot  ( epochs,loss )
    plt.plot  ( epochs,val_loss )
    plt.title ('Training and validation loss')

def show_accuracy(hist):
    acc = hist.history['accuracy']
    val_acc  = hist.history['val_accuracy']
    epochs   = range(len(acc))
    plt.figure()
    plt.plot  ( epochs,     acc )
    plt.plot  ( epochs, val_acc )
    plt.title ('Training and validation accuracy')

def encode(text, label):
    text = text.numpy().decode("utf-8")
    enc = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )
    return (
        enc["input_ids"],
        enc["attention_mask"],
        int(label.numpy())
    )

def tf_encode(text, label):
    input_ids, attn, y = tf.py_function(
        encode,
        inp=[text, label],
        Tout=[tf.int32, tf.int32, tf.int32]
    )
    input_ids.set_shape([MAX_LEN])
    attn.set_shape([MAX_LEN])
    y.set_shape([])
    return {"input_ids": input_ids, "attention_mask": attn}, y


In [3]:
imdb_df = pd.read_csv('data/IMDB Dataset.csv')
imdb_df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [4]:
train_reviews, test_reviews, train_labels, test_labels = train_test_split(
    imdb_df.review,
    imdb_df.sentiment,
    test_size=0.2,
    random_state=42,
    stratify=imdb_df.sentiment
)

train_reviews, val_reviews, train_labels, val_labels = train_test_split(
    train_reviews,
    train_labels,
    test_size=0.2,
    random_state=42,
    stratify=train_labels
)

train_labels = (train_labels == "positive").astype("int32")
test_labels  = (test_labels  == "positive").astype("int32")
val_labels  = (val_labels  == "positive").astype("int32")

train_ds = tf.data.Dataset.from_tensor_slices(
    (train_reviews.astype(str).to_numpy(), train_labels.to_numpy())
)

test_ds = tf.data.Dataset.from_tensor_slices(
    (test_reviews.astype(str).to_numpy(), test_labels.to_numpy())
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_reviews.astype(str).to_numpy(), val_labels.to_numpy())
)

In [5]:
model_name = "distilbert-base-uncased"
# model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
MAX_LEN = 128
BATCH = 4

In [6]:
train = train_ds.map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE).shuffle(20000).batch(BATCH).prefetch(tf.data.AUTOTUNE)

val = val_ds.map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE).shuffle(20000).batch(BATCH).prefetch(tf.data.AUTOTUNE)

test = test_ds.map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

In [ ]:
tf.keras.backend.clear_session()
gc.collect()

model = TFAutoModelForSequenceClassification.from_pretrained( model_name, num_labels=2)

loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1, min_lr=1e-7)
]

model.compile(optimizer='adam', loss=loss, metrics=['accuracy'])
history = model.fit(train, validation_data=val, epochs=2, callbacks=callbacks)
show_accuracy(history)
show_loss(history)

In [ ]:
tf.keras.backend.clear_session()
gc.collect()

model = TFAutoModelForSequenceClassification.from_pretrained( model_name, num_labels=2)

optimizer = Adam(learning_rate=2e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)


callbacks = [
    EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1, min_lr=1e-7)
]

model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])
history = model.fit(train, validation_data=val, epochs=2, callbacks=callbacks)
show_accuracy(history)
show_loss(history)

Some layers from the model checkpoint at distilbert-base-uncased were not used when initializing TFDistilBertForSequenceClassification: ['vocab_projector', 'activation_13', 'vocab_transform', 'vocab_layer_norm']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFDistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['dropout_19', 'classifier', 'pre_classifier']
You should probably TRAIN this model on a down-stream task to be able to use i

Epoch 1/2
1933/8000 [======>.......................] - ETA: 56:29 - loss: 0.4021 - accuracy: 0.8139

In [ ]:
labels_map = {0: "negative", 1: "positive"}

demo = test_ds.map(tf_encode).shuffle(20000).batch(BATCH)
x_batch, y_true = next(iter(demo))

logits = model(x_batch, training=False).logits
probs = tf.nn.softmax(logits, axis=1)
y_pred = tf.argmax(probs, axis=1)

texts = tokenizer.batch_decode(
    x_batch["input_ids"].numpy(),
    skip_special_tokens=True
)

for i in range(len(texts)):
    print("TEXT:")
    print(texts[i][:300])
    print("PREDICTED:", labels_map[int(y_pred[i])],
          f"(p={probs[i][y_pred[i]]:.3f})")
    print("TRUE:", labels_map[int(y_true[i])])
    print("-" * 60)


TEXT: the story of the bride fair is an amusing and engaging one, and it is to the filmmaker's credit that he sets out to portray rural minnesotans with the same respect ordinarily reserved for coast - dwellers. it is weird, though, to find an independent movie, the brainchild of a single person, that is

PREDICTED: negative (p=0.563) TRUE: negative

TEXT: a team varied between scully and mulder, two other scientists, a pilot, and the guy who plays bana on seinfeld, go up to an arctic research post where all members have died off by either killing each other or killing themselves. they discover there's a worm - a virus - that is parasitic to the point

PREDICTED: negative (p=0.563) TRUE: positive

TEXT: this was a popular movie probably because of the humor in it, the fast - moving story, an underdog character who shuts up all the loudmouths, etc. funny thing is, you probably couldn't make a movie with this title if you substituted anybody but " white " as anything else would be deemed racist by th

PREDICTED: negative (p=0.563) TRUE: negative
